# Nykaa Domain Support Agent — Task 5: Compare Chunking Strategies

This notebook evaluates the two ChromaDB collections created in Task 3:
- `nykaa_fixed_chunks`
- `nykaa_sentence_chunks`

It uses the same five or more test queries and manually defined relevant parent-document labels for both collections. Retrieved chunks are mapped to parent documents and deduplicated before Precision@3 and Recall@3 are calculated.

**Run this notebook from the same project directory that contains `chroma_db/`.**

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

CHROMA_DIR = Path.cwd() / "chroma_db"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3

FIXED_COLLECTION_NAME = "nykaa_fixed_chunks"
SENTENCE_COLLECTION_NAME = "nykaa_sentence_chunks"

if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        "chroma_db folder not found. Run Task 3 first and use the same project directory."
    )

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
fixed_collection = client.get_collection(FIXED_COLLECTION_NAME)
sentence_collection = client.get_collection(SENTENCE_COLLECTION_NAME)

embedding_model = SentenceTransformer(MODEL_NAME)

print("Fixed collection chunks:", fixed_collection.count())
print("Sentence collection chunks:", sentence_collection.count())
print("Embedding model loaded:", MODEL_NAME)


## 2. Define test queries and relevance labels

For each query, list the policy documents that are genuinely relevant to answering it. These labels are established before retrieval and are applied identically to both collections.

The `relevant_docs` values are filenames in the `knowledge_base/` folder. For example, a COD refund timeline question is labelled against `cod_refund.md`.

In [ ]:
TEST_QUERIES = [
    {
        "query": "How long does a cash-on-delivery refund take?",
        "relevant_docs": {"cod_refund.md"}
    },
    {
        "query": "Can I return a beauty product after delivery?",
        "relevant_docs": {"return_window.md"}
    },
    {
        "query": "What is the standard delivery time for domestic orders?",
        "relevant_docs": {"delivery_sla.md"}
    },
    {
        "query": "What are the conditions for exchanging a footwear size?",
        "relevant_docs": {"size_exchange.md", "return_window.md"}
    },
    {
        "query": "What should I do if payment was deducted but no order was created?",
        "relevant_docs": {"payment_failure.md", "escalation_matrix.md"}
    },
    {
        "query": "When is a reverse pickup arranged for a returned item?",
        "relevant_docs": {"reverse_pickup.md", "return_window.md"}
    }
]

print("Number of evaluation queries:", len(TEST_QUERIES))
for i, item in enumerate(TEST_QUERIES, start=1):
    print(f"{i}. {item['query']}")
    print("   Relevant documents:", sorted(item["relevant_docs"]))


## 3. Retrieve top-3 chunks from a collection

ChromaDB returns chunks, not unique documents. This function maps each retrieved chunk to its parent filename and removes duplicate filenames while preserving retrieval order.

We still use three result slots for Precision@3. If fewer than three unique parent documents are retrieved, the missing slots count as non-relevant. This convention is stated explicitly for consistent scoring.

In [ ]:
def retrieve_parent_documents(collection, query, top_k=TOP_K):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    ranked_parent_docs = []
    seen = set()

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for metadata, distance in zip(metadatas, distances):
        source = metadata["source"]

        if source not in seen:
            seen.add(source)
            ranked_parent_docs.append({
                "source": source,
                "cosine_similarity": 1 - float(distance)
            })

        # Stop once three unique parent documents have been collected.
        if len(ranked_parent_docs) == top_k:
            break

    return ranked_parent_docs


## 4. Calculate Precision@3 and Recall@3

- Precision@3 = relevant unique parent documents retrieved in the top 3 ÷ 3.
- Recall@3 = relevant unique parent documents retrieved in the top 3 ÷ total relevant parent documents in the relevance labels.

Each parent document is counted at most once.

In [ ]:
def evaluate_query(collection, query_item, strategy_name):
    query = query_item["query"]
    relevant_docs = query_item["relevant_docs"]

    retrieved = retrieve_parent_documents(collection, query, TOP_K)
    retrieved_docs = [item["source"] for item in retrieved]

    relevant_retrieved = [
        doc for doc in retrieved_docs if doc in relevant_docs
    ]

    relevant_count = len(relevant_retrieved)

    precision_at_3 = relevant_count / TOP_K
    recall_at_3 = (
        relevant_count / len(relevant_docs)
        if relevant_docs else 0.0
    )

    return {
        "strategy": strategy_name,
        "query": query,
        "relevant_docs": sorted(relevant_docs),
        "retrieved_docs": retrieved_docs,
        "relevant_retrieved": relevant_retrieved,
        "relevant_retrieved_count": relevant_count,
        "precision_at_3": precision_at_3,
        "recall_at_3": recall_at_3
    }


## 5. Evaluate both collections on the same queries

In [ ]:
all_results = []

for query_item in TEST_QUERIES:
    all_results.append(
        evaluate_query(
            fixed_collection,
            query_item,
            "Fixed-size with overlap"
        )
    )

    all_results.append(
        evaluate_query(
            sentence_collection,
            query_item,
            "Sentence-based"
        )
    )

results_df = pd.DataFrame(all_results)

display(
    results_df[
        [
            "strategy",
            "query",
            "relevant_docs",
            "retrieved_docs",
            "relevant_retrieved",
            "precision_at_3",
            "recall_at_3"
        ]
    ]
)


## 6. Show per-query arithmetic

This output makes the calculation visible for every query and both chunking strategies, as required by the capstone.

In [ ]:
for result in all_results:
    print("\\n" + "=" * 85)
    print("STRATEGY:", result["strategy"])
    print("QUERY:", result["query"])
    print("Relevant documents:", result["relevant_docs"])
    print("Retrieved unique parent documents:", result["retrieved_docs"])
    print("Relevant documents retrieved:", result["relevant_retrieved"])

    print(
        f"Precision@3 = {result['relevant_retrieved_count']}/3 "
        f"= {result['precision_at_3']:.3f}"
    )

    total_relevant = len(result["relevant_docs"])
    print(
        f"Recall@3 = {result['relevant_retrieved_count']}/{total_relevant} "
        f"= {result['recall_at_3']:.3f}"
    )


## 7. Compare average performance

The averages are calculated across the same evaluation queries for each collection. Use these results, along with the per-query arithmetic, to make the deployment recommendation.

In [ ]:
summary_df = (
    results_df
    .groupby("strategy")[["precision_at_3", "recall_at_3"]]
    .mean()
    .reset_index()
)

summary_df = summary_df.rename(columns={
    "precision_at_3": "mean_precision_at_3",
    "recall_at_3": "mean_recall_at_3"
})

display(summary_df)

fixed_summary = summary_df[
    summary_df["strategy"] == "Fixed-size with overlap"
].iloc[0]

sentence_summary = summary_df[
    summary_df["strategy"] == "Sentence-based"
].iloc[0]

print("\\nAVERAGE COMPARISON")
print(
    "Fixed-size: "
    f"Precision@3={fixed_summary['mean_precision_at_3']:.3f}, "
    f"Recall@3={fixed_summary['mean_recall_at_3']:.3f}"
)
print(
    "Sentence-based: "
    f"Precision@3={sentence_summary['mean_precision_at_3']:.3f}, "
    f"Recall@3={sentence_summary['mean_recall_at_3']:.3f}"
)


## 8. Generate a numbers-based recommendation

This cell compares mean Precision@3 and Recall@3. It recommends a strategy only when one is at least as good on both metrics and strictly better on one. If the metrics trade off or tie, it reports that the result is inconclusive and requires a reasoned decision.

In [ ]:
fixed_p = float(fixed_summary["mean_precision_at_3"])
fixed_r = float(fixed_summary["mean_recall_at_3"])
sentence_p = float(sentence_summary["mean_precision_at_3"])
sentence_r = float(sentence_summary["mean_recall_at_3"])

if (
    fixed_p >= sentence_p and fixed_r >= sentence_r
    and (fixed_p > sentence_p or fixed_r > sentence_r)
):
    recommendation = (
        "Recommend fixed-size chunking: it is at least as strong on both "
        "mean Precision@3 and mean Recall@3, and strictly better on one."
    )
elif (
    sentence_p >= fixed_p and sentence_r >= fixed_r
    and (sentence_p > fixed_p or sentence_r > fixed_r)
):
    recommendation = (
        "Recommend sentence-based chunking: it is at least as strong on both "
        "mean Precision@3 and mean Recall@3, and strictly better on one."
    )
else:
    recommendation = (
        "The metrics do not identify a clear winner because they trade off or tie. "
        "Review per-query results and choose based on the more important retrieval "
        "objective; report the trade-off rather than claiming one strategy wins."
    )

print(recommendation)


## 9. Save evaluation evidence

Save the per-query results and the average comparison as CSV files so they can be included in the GitHub repository and referenced in the README.

In [ ]:
output_dir = Path.cwd() / "results"
output_dir.mkdir(exist_ok=True)

per_query_path = output_dir / "task5_per_query_results.csv"
summary_path = output_dir / "task5_strategy_summary.csv"

results_df.to_csv(per_query_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Saved:", per_query_path.resolve())
print("Saved:", summary_path.resolve())
print("\\nPASS: Both chunking strategies were evaluated on the same queries.")
print("PASS: Parent documents were deduplicated before scoring.")
print("PASS: Per-query Precision@3 and Recall@3 were calculated.")
print("PASS: Average metrics and a recommendation were produced.")
